# Day 1: Infrastructure Foundation & Iceberg Storage Setup

This notebook simulates the local mock Iceberg catalog, creating the Bronze raw payload schema with audit metadata.

In [ ]:
!pip install pyspark pyiceberg duckdb splink

In [ ]:
import os
from pyspark.sql import SparkSession

# Setup local Iceberg warehouse path
warehouse_path = "/tmp/iceberg_warehouse"
os.makedirs(warehouse_path, exist_ok=True)

# Initialize Spark Session with Iceberg configurations
spark = SparkSession.builder \
    .appName("Day01_Iceberg_Setup") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", warehouse_path) \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark Session initialized with Iceberg catalog.")

In [ ]:
# Create Database / Namespace for Bronze layer
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.bronze")

# Create Bronze Table with audit metadata and raw payload
spark.sql("""
CREATE TABLE IF NOT EXISTS local.bronze.raw_fhir_payload (
    payload_id STRING,
    source_system_id STRING,
    raw_payload_json STRING,
    ingestion_timestamp TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(ingestion_timestamp))
""")

print("Iceberg Bronze table `raw_fhir_payload` created successfully.")

# Describe the table to verify schema
spark.sql("DESCRIBE local.bronze.raw_fhir_payload").show(truncate=False)